# Feature Extraction — Brain Invaders (bi2015a)

Loads preprocessed epochs from 01_preprocessing.ipynb and extracts feature sets for P300 classification.

## 1. Imports

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import mne
from scipy.signal import butter, sosfiltfilt
from pathlib import Path

mne.set_log_level('WARNING') # suppress MNE's verbose progress messages; change to 'INFO' to debug

## 2. Configuration

**Change SUBJECT to process a different participant.** All file paths and channel indices are derived automatically.

In [ ]:
# USER CONFIG 
SUBJECT     = 2   # participant number (1–43); avoid 1 and 27 (bad recordings)
_PROJECT_ROOT = Path().resolve()
DATA_ROOT   = _PROJECT_ROOT / "data" / "raw" / f"subject_{SUBJECT:02d}_csv"
OUTPUT_ROOT = _PROJECT_ROOT / "data" / "preprocessed"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)  

FLASH_DURATION_MS = {1: 110, 2: 80, 3: 50}   # From Table 2 of the bi2015a paper, session 1 slowest, session 3 fastest

SESSIONS = [1, 2, 3]

# Delta channels: posterior/centro-parietal — where the P300 slow wave is strongest
DELTA_CHANNELS = ['Cz', 'CP1', 'CP2', 'CP5', 'CP6', 'Pz', 'P3', 'P4', 'P8']
# Theta channels: frontal/fronto-central — where attentional theta bursts are strongest
THETA_CHANNELS = ['AFz', 'FC1', 'FC2', 'F3', 'F4', 'Cz']

# Bandpass filter parameters 
SFREQ      = 512.0   # amplifier sampling rate (must match preprocessing)
DELTA_LOW  = 0.5     # Hz — lower edge of delta band (0.5 avoids DC drift artefacts)
DELTA_HIGH = 3.0     # Hz — upper edge of delta band
THETA_LOW  = 3.0     # Hz — lower edge of theta band
THETA_HIGH = 7.0     # Hz — upper edge of theta band
BP_ORDER   = 4       # Butterworth filter order: steeper roll-off than 2, stable unlike 8+

# VERIFY FILES EXIST 
print(f"Data root      : {DATA_ROOT}")
print(f"Output root    : {OUTPUT_ROOT}")
print(f"Subject        : {SUBJECT}")
print(f"Delta channels : {DELTA_CHANNELS}  band={DELTA_LOW}–{DELTA_HIGH} Hz  order={BP_ORDER}")
print(f"Theta channels : {THETA_CHANNELS}  band={THETA_LOW}–{THETA_HIGH} Hz  order={BP_ORDER}")
print()
for s in SESSIONS:
    fif = OUTPUT_ROOT / f"subject_{SUBJECT:02d}_session_{s:02d}_epo.fif"
    status = 'OK' if fif.exists() else 'MISSING — run 01_preprocessing.ipynb first'
    print(f"  Session {s} ({FLASH_DURATION_MS[s]:3d} ms): [{status}]  {fif.name}")

## 3. Helper Functions

In [ ]:
def extract_erp(epoch_data, times, downsample_factor=16, channel_indices=None):
    """Crop to post-stimulus window (≥ 0 s), optionally restrict channels, downsample, flatten.
    epoch_data : (n_epochs, n_channels, n_times) → (n_epochs, n_ch * n_times_ds)
    """

    mask    = times >= 0 # exclude pre-stimulus baseline — contains no ERP signal
    cropped = epoch_data[:, :, mask]

    if channel_indices is not None:
        cropped = cropped[:, channel_indices, :]
    return cropped[:, :, ::downsample_factor].reshape(len(epoch_data), -1)


def bandpass_filter(data, sfreq, low, high, order=4):
    """Zero-phase Butterworth bandpass filter applied along the time axis.
    data : (n_epochs, n_channels, n_times)
    Returns filtered array of the same shape.
    """
    nyq = sfreq / 2.0
    # output='sos' avoids numerical instability of b/a form at low frequencies
    sos = butter(order, [low / nyq, high / nyq], btype='band', output='sos')
    # sosfiltfilt: forward + backward pass → zero phase delay → P300 peak timing preserved
    return sosfiltfilt(sos, data, axis=2)

## 4. Main Loop — Load, Extract, Save

Iterates over all three sessions for the selected subject.

In [ ]:
all_sessions = {}

for SESSION in SESSIONS:
    fif_path  = OUTPUT_ROOT / f"subject_{SUBJECT:02d}_session_{SESSION:02d}_epo.fif"
    save_path = OUTPUT_ROOT / f"subject_{SUBJECT:02d}_session_{SESSION:02d}_features.npz"
    flash_ms  = FLASH_DURATION_MS[SESSION]

    # 4a. Load 
    if not fif_path.exists():
        print(f"WARNING: {fif_path.name} not found — skipping session {SESSION}")
        continue

    epochs = mne.read_epochs(fif_path, verbose=False)

    if len(epochs) == 0:
        print(f"  WARNING: Subject {SUBJECT} Session {SESSION} has 0 epochs — skipping")
        continue
    if len(epochs) < 20:
        print(f"  WARNING: Subject {SUBJECT} Session {SESSION} has only {len(epochs)} epochs — skipping")
        continue

    # 4b. Extract raw data 
    eeg_picks    = mne.pick_types(epochs.info, eeg=True, stim=False)
    eeg_ch_names = [epochs.ch_names[i] for i in eeg_picks]
    eeg_info     = mne.pick_info(epochs.info, eeg_picks)

    cz_idx        = eeg_ch_names.index('Cz')
    delta_indices = [eeg_ch_names.index(ch) for ch in DELTA_CHANNELS] 
    theta_indices = [eeg_ch_names.index(ch) for ch in THETA_CHANNELS] 

    data   = epochs.get_data(picks='eeg')
    times  = epochs.times
    labels = epochs.metadata['target'].values

    # 4c. Extract features 
    features_erp = extract_erp(data, times)
    # Filter the full epoch before cropping — avoids edge artefacts at the stimulus onset boundary
    data_delta = bandpass_filter(data, SFREQ, DELTA_LOW, DELTA_HIGH, BP_ORDER)
    data_theta = bandpass_filter(data, SFREQ, THETA_LOW, THETA_HIGH, BP_ORDER)

    features_delta_wave = extract_erp(data_delta, times, channel_indices=delta_indices)
    features_theta_wave = extract_erp(data_theta, times, channel_indices=theta_indices)

    features_dt_wave    = np.hstack([features_delta_wave, features_theta_wave])

    # 4d. Print summary 
    n_target    = int(labels.sum())
    n_nontarget = len(labels) - n_target
    print(f"Session {SESSION} ({flash_ms} ms) — {len(epochs)} epochs ({n_target} target, {n_nontarget} non-target)")
    print(f"  ERP shape        : {features_erp.shape}")
    print(f"  Delta wave shape : {features_delta_wave.shape}")
    print(f"  Theta wave shape : {features_theta_wave.shape}")
    print(f"  DT wave shape    : {features_dt_wave.shape}")

    # 4e. Save features 
    np.savez(
        save_path,
        erp        = features_erp,
        delta_wave = features_delta_wave,
        theta_wave = features_theta_wave,
        dt_wave    = features_dt_wave,
        labels     = labels,
        flash_ms   = flash_ms,
        subject    = SUBJECT,
        session    = SESSION,
    )
    print(f"  Saved → {save_path}")
    print()

    all_sessions[SESSION] = dict(
        flash_ms      = flash_ms,
        data          = data, 
        times         = times,
        labels        = labels,
        eeg_info      = eeg_info,
        cz_idx        = cz_idx,
        delta_indices = delta_indices,
        theta_indices = theta_indices,
        data_delta    = data_delta,
        data_theta    = data_theta,
    )

## 5. Per-Session Visualisation
Three panels per session: broadband ERP at Cz, delta wave at Pz, theta wave at AFz. Target vs non-target averages — the red–blue gap is the discriminative signal the classifier exploits.

In [ ]:
# Positions of the representative electrodes within their channel-subset lists
pz_in_delta  = DELTA_CHANNELS.index('Pz') 
afz_in_theta = THETA_CHANNELS.index('AFz')

for SESSION in sorted(all_sessions):
    sess          = all_sessions[SESSION]
    flash_ms      = sess['flash_ms']
    data          = sess['data']
    times         = sess['times']
    labels        = sess['labels']
    cz_idx        = sess['cz_idx']
    delta_indices = sess['delta_indices']
    theta_indices = sess['theta_indices']
    data_delta    = sess['data_delta']
    data_theta    = sess['data_theta']

    t_mask  = labels == 1   # True for every target epoch row
    nt_mask = labels == 0   # True for every non-target epoch row

    # Resolve the 32-channel array index for Pz and AFz from their subset positions
    pz_idx  = delta_indices[pz_in_delta]
    afz_idx = theta_indices[afz_in_theta]

    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    fig.suptitle(
        f'Feature extraction — Subject {SUBJECT}, Session {SESSION} ({flash_ms} ms flash)',
        fontsize=13, fontweight='bold',
    )

    # Panel 1: ERP at Cz 
    ax = axes[0]
    ax.plot(times, data[t_mask,  cz_idx, :].mean(0) * 1e6,
            color='red',  label=f'Target (n={t_mask.sum()})')
    ax.plot(times, data[nt_mask, cz_idx, :].mean(0) * 1e6,
            color='blue', label=f'Non-target (n={nt_mask.sum()})')
    ax.axvline(0, color='black', linestyle='--', alpha=0.6, linewidth=0.9)   # stimulus onset
    ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title('ERP at Cz — Target vs Non-target')
    ax.legend(fontsize=9)

    # Panel 2: Delta bandpass waveform at Pz 
    ax = axes[1]
    ax.plot(times, data_delta[t_mask,  pz_idx, :].mean(0) * 1e6,
            color='red',  label=f'Target (n={t_mask.sum()})')
    ax.plot(times, data_delta[nt_mask, pz_idx, :].mean(0) * 1e6,
            color='blue', label=f'Non-target (n={nt_mask.sum()})')
    ax.axvline(0, color='black', linestyle='--', alpha=0.6, linewidth=0.9)
    ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f'Delta wave at Pz (0.5–3 Hz bandpass, order={BP_ORDER})')
    ax.legend(fontsize=9)

    # Panel 3: Theta bandpass waveform at AFz 
    ax = axes[2]
    ax.plot(times, data_theta[t_mask,  afz_idx, :].mean(0) * 1e6,
            color='red',  label=f'Target (n={t_mask.sum()})')
    ax.plot(times, data_theta[nt_mask, afz_idx, :].mean(0) * 1e6,
            color='blue', label=f'Non-target (n={nt_mask.sum()})')
    ax.axvline(0, color='black', linestyle='--', alpha=0.6, linewidth=0.9)
    ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f'Theta wave at AFz (3–7 Hz bandpass, order={BP_ORDER})')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()
    plt.close('all')

## 6. Cross-Session Comparison

3 × 3 grid: rows = ERP / delta / theta, columns = 110 ms / 80 ms / 50 ms flash duration.

In [ ]:
if len(all_sessions) < 3:
    print(f"Only {len(all_sessions)} session(s) loaded. All 3 are required for cross-session comparison.")
else:
    session_list = sorted(all_sessions.keys()) 

    pz_in_delta  = DELTA_CHANNELS.index('Pz')
    afz_in_theta = THETA_CHANNELS.index('AFz')

    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle(
        f'Bandpass waveform comparison across flash durations — Subject {SUBJECT}',
        fontsize=14, fontweight='bold',
    )

    row_labels = ['ERP at Cz (µV)', 'Delta wave at Pz (µV)', 'Theta wave at AFz (µV)']
    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=10, labelpad=8)  

    for col, SESSION in enumerate(session_list): 
        sess          = all_sessions[SESSION]
        flash_ms      = sess['flash_ms']
        data          = sess['data']
        times         = sess['times']
        labels        = sess['labels']
        cz_idx        = sess['cz_idx']
        delta_indices = sess['delta_indices']
        theta_indices = sess['theta_indices']
        data_delta    = sess['data_delta']
        data_theta    = sess['data_theta']

        pz_idx  = delta_indices[pz_in_delta]
        afz_idx = theta_indices[afz_in_theta]

        t_mask  = labels == 1   # select target epochs
        nt_mask = labels == 0   # select non-target epochs

        ax = axes[0, col]
        ax.plot(times, data[t_mask,  cz_idx, :].mean(0) * 1e6,
                color='red',  label='Target',     linewidth=1.2)
        ax.plot(times, data[nt_mask, cz_idx, :].mean(0) * 1e6,
                color='blue', label='Non-target', linewidth=1.2)
        ax.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=0.8)
        ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
        ax.set_title(f'{flash_ms} ms flash', fontweight='bold') 
        ax.set_xlabel('Time (s)')
        if col == 0:
            ax.legend(fontsize=7)  

        ax = axes[1, col]
        ax.plot(times, data_delta[t_mask,  pz_idx, :].mean(0) * 1e6,
                color='red',  label='Target',     linewidth=1.2)
        ax.plot(times, data_delta[nt_mask, pz_idx, :].mean(0) * 1e6,
                color='blue', label='Non-target', linewidth=1.2)
        ax.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=0.8)
        ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
        ax.set_xlabel('Time (s)')
        if col == 0:
            ax.legend(fontsize=7)

        ax = axes[2, col]
        ax.plot(times, data_theta[t_mask,  afz_idx, :].mean(0) * 1e6,
                color='red',  label='Target',     linewidth=1.2)
        ax.plot(times, data_theta[nt_mask, afz_idx, :].mean(0) * 1e6,
                color='blue', label='Non-target', linewidth=1.2)
        ax.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=0.8)
        ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
        ax.set_xlabel('Time (s)')
        if col == 0:
            ax.legend(fontsize=7)

    plt.tight_layout()
    plt.show()
    plt.close('all')